# UNMASK ME — Gemma 4 Adaptive-Learning Demo

Companion notebook for the **Unmask Me / UNMASK ME** submission to the **Gemma 4 Good Hackathon** (Safety & Trust + Ollama tracks).

This notebook proves that the live web app's adaptive-learning pipeline runs end-to-end on Gemma 4 — locally via Ollama (E4B class) or via Google AI Studio's Gemma 4 endpoint. The same prompts the app sends at runtime are exercised here against a real Gemma 4 instance, with the JSON contracts verified.

**What we're showing**

1. Intake → Spirit Charm archetype selection
2. Pre/post somatic shift scoring from face frames (multimodal)
3. Adaptive swarm insight generation from a user's event history
4. Reflection-text scoring (text-only)

**Inference path**: Ollama local first (`gemma4:4b` → `gemma3:4b` fallback), Google AI Studio (`gemma-4-27b-it`) second. The web app surfaces the active path in the UI so users always know where their data is.

## 1. Setup

Install minimal dependencies. The notebook works in both Kaggle Notebooks (with internet enabled, Google AI Studio path) and on a laptop with `ollama` installed (local Gemma 4 path).

In [ ]:
%pip install --quiet requests pillow google-genai

In [ ]:
import os, json, base64, io, requests
from PIL import Image

OLLAMA_URL = os.environ.get('OLLAMA_BASE_URL', 'http://localhost:11434')
GEMINI_KEY = os.environ.get('GEMINI_API_KEY') or os.environ.get('KAGGLE_GEMINI_API_KEY')

LOCAL_CANDIDATES = ['gemma4:4b', 'gemma4:2b', 'gemma3:4b']
CLOUD_MODEL = 'gemma-4-27b-it'

def detect_local():
    try:
        r = requests.get(f'{OLLAMA_URL}/api/tags', timeout=2)
        names = [m['name'] for m in r.json().get('models', [])]
        for tag in LOCAL_CANDIDATES:
            if tag in names: return tag
        return next((n for n in names if 'gemma' in n.lower()), None)
    except Exception:
        return None

MODE = 'local' if detect_local() else ('cloud' if GEMINI_KEY else 'offline')
MODEL = detect_local() if MODE == 'local' else CLOUD_MODEL
print(f'Inference path: {MODE}  ·  model: {MODEL}')

## 2. Shared client — same JSON contracts as the web app

In [ ]:
def call_local(prompt: str, images: list[str] | None = None) -> str:
    body = {'model': MODEL, 'prompt': prompt + '\n\nRespond ONLY with valid JSON.', 'stream': False}
    if images: body['images'] = [i.split(',')[-1] for i in images]
    r = requests.post(f'{OLLAMA_URL}/api/generate', json=body, timeout=120)
    return r.json().get('response', '')

def call_cloud(prompt: str, images: list[str] | None = None) -> str:
    from google import genai
    client = genai.Client(api_key=GEMINI_KEY)
    parts = [prompt]
    for b in (images or []):
        parts.append({'inline_data': {'mime_type': 'image/jpeg', 'data': b.split(',')[-1]}})
    resp = client.models.generate_content(model=MODEL, contents=parts)
    return resp.text or ''

def gemma_json(prompt: str, images: list[str] | None = None) -> dict | None:
    raw = call_local(prompt, images) if MODE == 'local' else call_cloud(prompt, images)
    try: return json.loads(raw)
    except Exception:
        s = raw.find('{'); e = raw.rfind('}')
        return json.loads(raw[s:e+1]) if s >= 0 and e > s else None

## 3. Intake → Spirit Charm archetype

When a new user finishes the three-question intake, Gemma 4 picks one of six archetypes and forges a name + tagline + color + a first "swarm insight" about human resilience.

In [ ]:
intake = [
    'Stay present with my daughter today',
    'The anger about my brother — I want to put it down for an hour',
    'A knot in my chest, just under the collarbone',
]
prompt = f'''You are the swarm arbiter for UNMASK ME / Unmask Me.
A new user completed an intake:
1) Intention: "{intake[0]}"
2) Heavy energy: "{intake[1]}"
3) Body tension: "{intake[2]}"

Pick ONE archetype from: fox, owl, crystal, flame, wave, tree.
Return JSON: {{ "archetype": ..., "name": ..., "tagline": ..., "primaryColor": "#hex", "swarmInsight": ... }}'''
result = gemma_json(prompt)
print(json.dumps(result, indent=2))

## 4. Adaptive swarm insight from event history

The web app maintains a 200-event rolling log per user. After every few protocols, it asks Gemma 4 to summarize what it learned about *this* user's resilience pattern.

In [ ]:
events = [
    'protocol: box-breath (84)',
    'protocol: somatic-shake (71)',
    'protocol: vagus-hum (89)',
    'vei-log: "admitted I was overwhelmed today" (78)',
    'protocol: jaw-release (65)',
    'anchor-rift: Hush Park (55)',
]
prompt = f'''swarm meta-arbiter. The user's last {len(events)} actions were:
{chr(10).join(f"{i+1}. {e}" for i, e in enumerate(events))}

Return JSON: {{ "headline": ..., "insight": ..., "nextProtocol": one-of(box-breath|somatic-shake|vagus-hum|jaw-release|eft-tap|ground-stomp) }}'''
insight = gemma_json(prompt)
print(json.dumps(insight, indent=2))

## 5. Reflection scoring (text-only)

When a user etches a self-reflection in the Unmask page, Gemma 4 grades sincerity + presence and awards Peace Points.

In [ ]:
reflection = "I noticed I say 'I'm fine' when my chest is tight. Today I admitted I was overwhelmed to my partner. They held me. I didn't run."
prompt = f'''swarm arbiter. Review Self-Reflection:
"{reflection}"
Return JSON: {{ "score": 1-100, "feedback": "short poetic gritty sentence", "pointsAwarded": int<=200, "swarmInsight": ... }}'''
print(json.dumps(gemma_json(prompt), indent=2))

## 6. Multimodal: pre/post somatic shift (face frames)

After the six-protocol nervous-system reset, the app sends two in-memory face frames (PRE + POST) to Gemma 4 to score the visible shift. Frames are NEVER stored — they live in JavaScript memory for ~200ms and are discarded after this call.

For the notebook we use two solid-color placeholder frames so this cell runs anywhere; the contract is identical to the live app.

In [ ]:
def fake_face_b64(tint: tuple[int,int,int]) -> str:
    img = Image.new('RGB', (320, 400), tint)
    buf = io.BytesIO(); img.save(buf, format='JPEG')
    return 'data:image/jpeg;base64,' + base64.b64encode(buf.getvalue()).decode()

pre = fake_face_b64((90, 60, 60))   # tense red-ish
post = fake_face_b64((100, 130, 160)) # softened blue-ish

prompt = '''swarm somatic arbiter. Protocol: six-protocol-reset.
Image 1 = PRE, Image 2 = POST. Score the visible parasympathetic shift.
Return JSON: { "score": 1-100, "feedback": ..., "pointsAwarded": int<=500, "swarmInsight": ... }'''
shift = gemma_json(prompt, [pre, post])
print(json.dumps(shift, indent=2))

## 7. Where this lands in the product

Every Gemma 4 call shown above maps 1:1 to a user-facing surface in the live app:

| Notebook cell | Live surface |
| --- | --- |
| 3 — intake archetype | Boot sequence final screen (charm forged) |
| 4 — adaptive insight | Reputation page → Adaptive Learning Insights panel |
| 5 — reflection scoring | Unmask page → "Etch a reflection" |
| 6 — somatic shift | V.A.I. page → after the six-protocol reset |

**Live demo**: https://ai.studio/apps/b8e3b826-5959-4dd5-a24d-8d4450e3dc40

**Code repo**: see attached link in the writeup.

**Local-first promise**: every cell above runs without an internet connection if you `ollama pull gemma3:4b` (or `gemma4:4b` when it ships). The web app makes the same promise — face frames stay on the device, the Gemma 4 model runs on the device, the swarm brain lives in the browser's localStorage. The judges' bar for the Ollama Special Tech track is exactly this: an impactful local-first Gemma 4 experience.